In [27]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Iterator, List, Optional
from icalendar import Calendar

# OOP

**Fred Brosman**

*13.04.2026*

## Ülesanne 1. Üliõpilaste hindamissüsteem

Selles ülesandes loon süsteemi, millega saab hallata üliõpilasi, nende aineid ja hindeid.

Lahenduses kasutan:
- klassi `Student`, mis hoiab ühe üliõpilase andmeid
- klassi `GradeSystem`, mis haldab kõiki üliõpilasi
- eraldi meetodeid hinnete lisamiseks, keskmise arvutamiseks ja tulemuste analüüsimiseks

Programm peab täitma järgmised nõuded:
1. leidma 3 parimat tudengit keskmise hinde põhjal
2. arvutama iga aine keskmise hinde ja sorteerima ained parimast alates
3. leidma tudengid, kellel on vähemalt ühes aines hinne 1 või 2

## 1.1 Klass Student

See klass kirjeldab ühte üliõpilast.
Igal üliõpilasel on ID, nimi ja hinnete sõnastik.

In [28]:
@dataclass
class Student:
    """
    Klass ühe üliõpilase andmete hoidmiseks.

    Attributes
    ----------
    id : int
        Üliõpilase ID.
    name : str
        Üliõpilase nimi.
    grades : dict
        Sõnastik, kus võtmed on ainete nimed ja väärtused on hinded.
    """

    id: int
    name: str
    grades: dict = field(default_factory=dict)

    def add_grade(self, course, grade):
        """
        Lisab üliõpilasele hinde kindlas aines.

        Parameters
        ----------
        course : str
            Aine nimi.
        grade : int
            Hinne vahemikus 1 kuni 5.
        """
        if grade < 1 or grade > 5:
            raise ValueError("Hinne peab olema vahemikus 1 kuni 5.")
        self.grades[course] = grade

    def get_average(self):
        """
        Arvutab üliõpilase kõikide hinnete keskmise.

        Returns
        -------
        float
            Keskmine hinne. Kui hindeid ei ole, tagastatakse 0.
        """
        if not self.grades:
            return 0
        return sum(self.grades.values()) / len(self.grades)

## 1.2 Klass GradeSystem

See klass haldab kõiki üliõpilasi ja võimaldab teha ülesandes nõutud analüüse.

In [29]:
@dataclass
class GradeSystem:
    """
    Klass üliõpilaste hindamissüsteemi haldamiseks.

    Attributes
    ----------
    students : list
        Nimekiri Student tüüpi objektidest.
    """

    students: list = field(default_factory=list)

    def add_student(self, student):
        """
        Lisab süsteemi ühe üliõpilase.

        Parameters
        ----------
        student : Student
            Süsteemi lisatav üliõpilane.
        """
        self.students.append(student)

    def get_top_students(self, count=3):
        """
        Leiab parimad tudengid keskmise hinde põhjal.

        Parameters
        ----------
        count : int
            Mitu tudengit tagastatakse.

        Returns
        -------
        list
            Nimekiri tuplena kujul (nimi, id, keskmine_hinne).
        """
        sorditud = sorted(
            self.students,
            key=lambda student: student.get_average(),
            reverse=True
        )

        tulemus = []
        for student in sorditud[:count]:
            tulemus.append((student.name, student.id, round(student.get_average(), 2)))
        return tulemus

    def get_course_averages(self):
        """
        Arvutab iga aine keskmise hinde.

        Returns
        -------
        list
            Nimekiri tuplena kujul (aine, keskmine_hinne),
            sordituna parimast ainest alates.
        """
        aine_hinded = {}

        for student in self.students:
            for course, grade in student.grades.items():
                if course not in aine_hinded:
                    aine_hinded[course] = []
                aine_hinded[course].append(grade)

        keskmised = []
        for course, grades in aine_hinded.items():
            keskmine = sum(grades) / len(grades)
            keskmised.append((course, round(keskmine, 2)))

        return sorted(keskmised, key=lambda x: x[1], reverse=True)

    def get_students_with_low_grades(self):
        """
        Leiab tudengid, kellel on vähemalt ühes aines hinne 1 või 2.

        Returns
        -------
        list
            Nimekiri tuplena kujul (nimi, [ained]).
        """
        tulemus = []

        for student in self.students:
            madalad_ained = []

            for course, grade in student.grades.items():
                if grade == 1 or grade == 2:
                    madalad_ained.append(course)

            if madalad_ained:
                tulemus.append((student.name, madalad_ained))

        return tulemus

## 1.3 Abifunktsioonid tulemuste väljastamiseks

Järgmised funktsioonid aitavad väljastada tulemusi selgemal kujul.

In [30]:
def prindi_parimad_tudengid(parimad):
    """
    Väljastab 3 parimat tudengit.

    Parameters
    ----------
    parimad : list
        Nimekiri kujul (nimi, id, keskmine_hinne).
    """
    print("3 parimat tudengit:")
    for nimi, tudengi_id, keskmine in parimad:
        print(f"Nimi: {nimi}, ID: {tudengi_id}, keskmine hinne: {keskmine}")


def prindi_aine_keskmised(aine_keskmised):
    """
    Väljastab ainete keskmised hinded.

    Parameters
    ----------
    aine_keskmised : list
        Nimekiri kujul (aine, keskmine_hinne).
    """
    print("\nAinete keskmised hinded:")
    for aine, keskmine in aine_keskmised:
        print(f"{aine}: {keskmine}")


def prindi_madalad_hinded(madalad_hinded):
    """
    Väljastab tudengid, kellel esineb madal hinne.

    Parameters
    ----------
    madalad_hinded : list
        Nimekiri kujul (nimi, [ained]).
    """
    print("\nTudengid, kellel on vähemalt ühes aines hinne 1 või 2:")
    for nimi, ained in madalad_hinded:
        print(f"{nimi}: {', '.join(ained)}")

## 1.4 Testandmete loomine

Selles plokis loon mõned näidisüliõpilased ja lisan neile hinded.

In [31]:
student1 = Student(1, "Mari")
student1.add_grade("Matemaatika", 5)
student1.add_grade("Programmeerimine", 4)
student1.add_grade("Füüsika", 5)

student2 = Student(2, "Juhan")
student2.add_grade("Matemaatika", 3)
student2.add_grade("Programmeerimine", 2)
student2.add_grade("Füüsika", 4)

student3 = Student(3, "Kati")
student3.add_grade("Matemaatika", 5)
student3.add_grade("Programmeerimine", 5)
student3.add_grade("Füüsika", 4)

student4 = Student(4, "Rasmus")
student4.add_grade("Matemaatika", 1)
student4.add_grade("Programmeerimine", 3)
student4.add_grade("Füüsika", 2)

student5 = Student(5, "Liis")
student5.add_grade("Matemaatika", 4)
student5.add_grade("Programmeerimine", 5)
student5.add_grade("Füüsika", 5)

## 1.5 Hindamissüsteemi loomine

Järgmises plokis lisan kõik loodud tudengid süsteemi.

In [32]:
system = GradeSystem()

system.add_student(student1)
system.add_student(student2)
system.add_student(student3)
system.add_student(student4)
system.add_student(student5)

## 1.6 Tulemuste arvutamine ja väljastamine

Selles plokis käivitan kõik ülesandes nõutud funktsionaalsused ja väljastan tulemused.

In [33]:
parimad = system.get_top_students()
aine_keskmised = system.get_course_averages()
madalad_hinded = system.get_students_with_low_grades()

prindi_parimad_tudengid(parimad)
prindi_aine_keskmised(aine_keskmised)
prindi_madalad_hinded(madalad_hinded)

3 parimat tudengit:
Nimi: Mari, ID: 1, keskmine hinne: 4.67
Nimi: Kati, ID: 3, keskmine hinne: 4.67
Nimi: Liis, ID: 5, keskmine hinne: 4.67

Ainete keskmised hinded:
Füüsika: 4.0
Programmeerimine: 3.8
Matemaatika: 3.6

Tudengid, kellel on vähemalt ühes aines hinne 1 või 2:
Juhan: Programmeerimine
Rasmus: Matemaatika, Füüsika


## Ülesanne 2. iCalendar tunniplaani analüüs OOP-iga

Selles ülesandes loen tunniplaani andmed `.ics` failist ja salvestan need
klasside eksemplaridesse.

Kasutan kolme põhiklassi:
- `Event` – üks sündmus
- `Course` – üks õppeaine koos sündmustega
- `Timetable` – kogu tunniplaan

Lahenduses kasutan objektorienteeritud programmeerimist, kapseldamist,
valideerimist, tüübimärgendusi ja docstring'e.

## 2.2 Klass Event

See klass kirjeldab ühte `.ics` faili sündmust.
Siin kontrollin ka, et sündmuse nimi ei oleks tühi ja lõpuaeg oleks algusajast hilisem.

In [34]:
from dataclasses import dataclass, field
from datetime import datetime
from typing import Optional


@dataclass
class Event:
    """
    Klass ühe tunniplaani sündmuse hoidmiseks.
    """

    dtstart: datetime
    dtend: datetime
    location: Optional[str] = None
    description: Optional[str] = None
    __summary: str = field(init=False, repr=False)

    def __init__(
        self,
        summary: str,
        dtstart: datetime,
        dtend: datetime,
        location: Optional[str] = None,
        description: Optional[str] = None
    ) -> None:
        """
        Loob uue sündmuse.
        """
        if not summary or not summary.strip():
            raise ValueError("Sündmuse nimi ei tohi olla tühi.")

        if dtend <= dtstart:
            raise ValueError("Sündmuse lõpuaeg peab olema algusajast hilisem.")

        self.__summary = summary.strip()
        self.dtstart = dtstart
        self.dtend = dtend
        self.location = location.strip() if location and location.strip() else None
        self.description = description.strip() if description and description.strip() else None

    @property
    def summary(self) -> str:
        """
        Tagastab sündmuse nime.
        """
        return self.__summary

    def duration_hours(self) -> float:
        """
        Arvutab sündmuse kestuse tundides.
        """
        return (self.dtend - self.dtstart).total_seconds() / 3600

    def __str__(self) -> str:
        """
        Tagastab sündmuse loetava tekstiesituse.
        """
        koht = self.location if self.location else "Asukoht puudub"
        return (
            f"{self.summary} | "
            f"{self.dtstart.strftime('%d.%m.%Y %H:%M')} - "
            f"{self.dtend.strftime('%H:%M')} | "
            f"{koht}"
        )

## 2.3 Klass Course

See klass koondab ühe õppeaine kõik sündmused.

In [35]:
@dataclass
class Course:
    """
    Klass ühe õppeaine andmete hoidmiseks.

    Attributes
    ----------
    code : str
        Õppeaine kood.
    teacher : str
        Õppejõu nimi.
    events : List[Event]
        Selle ainega seotud sündmused.
    """

    code: str
    teacher: str
    events: List[Event] = field(default_factory=list)
    __name: str = field(init=False, repr=False)

    def __init__(
        self,
        code: str,
        name: str,
        teacher: str,
        events: Optional[List[Event]] = None
    ) -> None:
        """
        Loob uue õppeaine.

        Parameters
        ----------
        code : str
            Õppeaine kood.
        name : str
            Õppeaine nimi.
        teacher : str
            Õppejõu nimi.
        events : Optional[List[Event]]
            Õppeaine sündmuste nimekiri.

        Raises
        ------
        ValueError
            Kui aine nimi on tühi.
        """
        if not name or not name.strip():
            raise ValueError("Õppeaine nimi ei tohi olla tühi.")

        self.code = code.strip() if code and code.strip() else "TUNDMATU"
        self.__name = name.strip()
        self.teacher = teacher.strip() if teacher and teacher.strip() else "Tundmatu"
        self.events = events[:] if events else []

    @property
    def name(self) -> str:
        """
        Tagastab õppeaine nime.

        Returns
        -------
        str
            Õppeaine nimi.
        """
        return self.__name

    def add_event(self, event: Event) -> None:
        """
        Lisab ainele ühe sündmuse.

        Parameters
        ----------
        event : Event
            Lisatav sündmus.
        """
        self.events.append(event)

    def event_count(self) -> int:
        """
        Tagastab aine sündmuste arvu.

        Returns
        -------
        int
            Sündmuste arv.
        """
        return len(self.events)

    def total_hours(self) -> float:
        """
        Arvutab aine kõigi sündmuste kogukestuse tundides.

        Returns
        -------
        float
            Kogukestus tundides.
        """
        return sum(event.duration_hours() for event in self.events)

    def __str__(self) -> str:
        """
        Tagastab õppeaine loetava tekstiesituse.

        Returns
        -------
        str
            Õppeaine tekstina.
        """
        return (
            f"{self.code} - {self.name} | "
            f"Õppejõud: {self.teacher} | "
            f"Sündmusi: {len(self.events)}"
        )

## 2.4 Klass Timetable

See klass hoiab kogu tunniplaani ja sisaldab meetodit `.ics` faili lugemiseks.

In [36]:
@dataclass
class Timetable:
    """
    Klass kogu tunniplaani hoidmiseks ja analüüsimiseks.

    Attributes
    ----------
    courses : List[Course]
        Kõik tunniplaanis olevad õppeained.
    """

    courses: List[Course] = field(default_factory=list)

    def __str__(self) -> str:
        """
        Tagastab tunniplaani lühikirjelduse.

        Returns
        -------
        str
            Tunniplaani kirjeldus.
        """
        return (
            f"Tunniplaan: {len(self.courses)} õppeainet, "
            f"{self.event_count()} sündmust"
        )

    def __iter__(self) -> Iterator[Event]:
        """
        Võimaldab käia kõik sündmused for-tsükliga läbi.

        Yields
        ------
        Event
            Järgmine sündmus tunniplaanis.
        """
        for course in self.courses:
            for event in course.events:
                yield event

    def add_course(self, course: Course) -> None:
        """
        Lisab tunniplaani ühe õppeaine.

        Parameters
        ----------
        course : Course
            Lisatav õppeaine.
        """
        self.courses.append(course)

    def all_events(self) -> List[Event]:
        """
        Tagastab kõik sündmused ühe nimekirjana.

        Returns
        -------
        List[Event]
            Kõik sündmused.
        """
        events = []
        for course in self.courses:
            events.extend(course.events)
        return events

    def event_count(self) -> int:
        """
        Tagastab kõigi sündmuste koguarvu.

        Returns
        -------
        int
            Sündmuste arv.
        """
        return len(self.all_events())

    def total_hours(self) -> float:
        """
        Arvutab kogu tunniplaani tundide koguarvu.

        Returns
        -------
        float
            Kõigi sündmuste kogukestus tundides.
        """
        return sum(event.duration_hours() for event in self.all_events())

    def meetings_per_course(self) -> dict[str, int]:
        """
        Tagastab, mitu kohtumist on igas aines.

        Returns
        -------
        dict[str, int]
            Sõnastik kujul {aine_nimi: kohtumiste_arv}.
        """
        tulemus = {}

        for course in self.courses:
            tulemus[course.name] = course.event_count()

        return tulemus

    def courses_by_teacher(self) -> dict[str, list[str]]:
        """
        Tagastab ainete nimekirja õppejõu järgi.

        Returns
        -------
        dict[str, list[str]]
            Sõnastik kujul {õppejõud: [ained]}.
        """
        tulemus = {}

        for course in self.courses:
            if course.teacher not in tulemus:
                tulemus[course.teacher] = []

            tulemus[course.teacher].append(course.name)

        for teacher in tulemus:
            tulemus[teacher] = sorted(set(tulemus[teacher]))

        return tulemus

    def next_two_courses(self) -> list[str]:
        """
        Leiab kaks erinevat ainet, mida õpetatakse kõige lähiajal.

        Returns
        -------
        list[str]
            Kahe lähima aine nimekiri.
        """
        koik_sundmused = []

        for course in self.courses:
            for event in course.events:
                koik_sundmused.append((course.name, event))

        if not koik_sundmused:
            return []

        esimene_aeg = koik_sundmused[0][1].dtstart

        if esimene_aeg.tzinfo is not None:
            praegu = datetime.now(esimene_aeg.tzinfo)
        else:
            praegu = datetime.now()

        tulevased = [
            (course_name, event)
            for course_name, event in koik_sundmused
            if event.dtstart >= praegu
        ]

        tulevased.sort(key=lambda item: item[1].dtstart)

        ained = []
        for course_name, event in tulevased:
            if course_name not in ained:
                ained.append(course_name)

            if len(ained) == 2:
                break

        return ained

    @staticmethod
    def ensure_datetime(value) -> datetime:
        """
        Teisendab .ics failist loetud väärtuse vajadusel datetime kujule.

        Parameters
        ----------
        value
            .ics failist loetud kuupäev või kuupäev-kellaaeg.

        Returns
        -------
        datetime
            Datetime kujul väärtus.
        """
        if isinstance(value, datetime):
            return value

        return datetime.combine(value, datetime.min.time())

    @staticmethod
    def extract_teacher(description: Optional[str]) -> str:
        """
        Proovib kirjeldusest leida õppejõu nime.

        Parameters
        ----------
        description : Optional[str]
            Sündmuse kirjeldus.

        Returns
        -------
        str
            Õppejõu nimi või 'Tundmatu'.
        """
        if not description:
            return "Tundmatu"

        for rida in description.splitlines():
            puhas = rida.strip()
            lower = puhas.lower()

            if lower.startswith("õppejõud:") or lower.startswith("teacher:"):
                osad = puhas.split(":", 1)
                if len(osad) == 2 and osad[1].strip():
                    return osad[1].strip()

        return "Tundmatu"

    @staticmethod
    def extract_course_info(summary: str, description: Optional[str]) -> tuple[str, str, str]:
        """
        Proovib summary ja description väljadest leida aine koodi, nime ja õppejõu.

        Parameters
        ----------
        summary : str
            Sündmuse nimi.
        description : Optional[str]
            Sündmuse kirjeldus.

        Returns
        -------
        tuple[str, str, str]
            (aine_kood, aine_nimi, õppejõud)
        """
        code = summary.strip()
        name = summary.strip()
        teacher = Timetable.extract_teacher(description)

        if " - " in summary:
            vasak, parem = summary.split(" - ", 1)
            if vasak.strip():
                code = vasak.strip()
            if parem.strip():
                name = parem.strip()

        return code, name, teacher

    @classmethod
    def from_ical(cls, filename: str) -> "Timetable":
        """
        Loeb tunniplaani .ics failist ja tagastab Timetable objekti.

        Parameters
        ----------
        filename : str
            Faili nimi või tee.

        Returns
        -------
        Timetable
            Täidetud tunniplaani objekt.
        """
        with open(filename, "rb") as f:
            cal = Calendar.from_ical(f.read())

        courses_by_key = {}

        for component in cal.walk():
            if component.name != "VEVENT":
                continue

            summary = str(component.get("summary", "")).strip()
            dtstart = cls.ensure_datetime(component.get("dtstart").dt)
            dtend = cls.ensure_datetime(component.get("dtend").dt)
            location = str(component.get("location")).strip() if component.get("location") else None
            description = str(component.get("description")).strip() if component.get("description") else None

            event = Event(
                summary=summary,
                dtstart=dtstart,
                dtend=dtend,
                location=location,
                description=description
            )

            code, name, teacher = cls.extract_course_info(summary, description)
            key = (code, name, teacher)

            if key not in courses_by_key:
                courses_by_key[key] = Course(
                    code=code,
                    name=name,
                    teacher=teacher,
                    events=[]
                )

            courses_by_key[key].add_event(event)

        return cls(list(courses_by_key.values()))

## 2.5 Tunniplaani laadimine

Selles plokis loen andmed `.ics` failist ja kontrollin, et klassid töötavad õigesti.

In [37]:
tunniplaan = Timetable.from_ical("tunniplaan.ics")

print(tunniplaan)
print()

for course in tunniplaan.courses:
    print(course)

Tunniplaan: 11 õppeainet, 51 sündmust

RAM0620 - Programmeerimine | Õppejõud: Maarika Virkunen | Sündmusi: 4
RAM0620 - Programmeerimine | Õppejõud: Natalja Ivleva | Sündmusi: 4
NTR0450 - Inseneriinformaatika | Õppejõud: Valeria Juštšenko | Sündmusi: 4
NTR0450 - Inseneriinformaatika | Õppejõud: Žanna Gratšjova | Sündmusi: 3
RAM0800 - Digitaalloogika ja -süsteemid | Õppejõud: Oleg Shvets | Sündmusi: 3
RAM0800 - Digitaalloogika ja -süsteemid | Õppejõud: Monika Jänis | Sündmusi: 6
RAE1100 - Rakendusfüüsika | Õppejõud: Maarika Virkunen | Sündmusi: 3
RAE1100 - Rakendusfüüsika | Õppejõud: Sergei Pavlov | Sündmusi: 5
RAM0810 - Operatsioonisüsteemide administreerimine | Õppejõud: Oleg Shvets | Sündmusi: 5
EVK0033 - Eesti keel III | Õppejõud: Valentina Limonova | Sündmusi: 8
EVR0320 - Rohetehnoloogiate baaskursus | Õppejõud: Dmitri Nikitin | Sündmusi: 6


## 2.6 Kõikide sündmuste kuvamine

Järgmises plokis väljastan kõik tunniplaanis olevad sündmused.

In [38]:
for event in tunniplaan.all_events():
    print(event)

RAM0620 - Programmeerimine | 07.02.2026 08:45 - 10:15 | VK1-42
RAM0620 - Programmeerimine | 07.03.2026 12:45 - 14:15 | VK1-40
RAM0620 - Programmeerimine | 08.03.2026 12:45 - 14:15 | VK1-40
RAM0620 - Programmeerimine | 22.03.2026 12:45 - 14:15 | VK1-40
RAM0620 - Programmeerimine | 21.03.2026 07:00 - 10:15 | VK1-28
RAM0620 - Programmeerimine | 04.04.2026 10:00 - 13:15 | VK1-28
RAM0620 - Programmeerimine | 18.04.2026 10:00 - 13:15 | VK1-28
RAM0620 - Programmeerimine | 22.02.2026 12:45 - 16:00 | VK1-28
NTR0450 - Inseneriinformaatika | 07.03.2026 11:00 - 12:30 | VK1-40
NTR0450 - Inseneriinformaatika | 07.02.2026 11:00 - 14:15 | VK1-40
NTR0450 - Inseneriinformaatika | 21.03.2026 12:45 - 14:15 | VK1-40
NTR0450 - Inseneriinformaatika | 18.04.2026 13:30 - 15:00 | VK1-40
NTR0450 - Inseneriinformaatika | 18.04.2026 06:00 - 09:15 | VK1-27
NTR0450 - Inseneriinformaatika | 04.04.2026 13:30 - 15:00 | VK1-27
NTR0450 - Inseneriinformaatika | 03.05.2026 10:00 - 13:15 | VK1-27
RAM0800 - Digitaalloogika j

## 2.7 Statistikafunktsioonid

Järgmisena lisan klassi `Timetable` sisse samad analüüsid, mis olid eelmises andmestruktuuride ülesandes punktides a–d.

In [39]:
tunniplaan = Timetable.from_ical("tunniplaan.ics")

print(tunniplaan)

print("\na) Mitu kohtumist on igas aines:")
for aine, arv in tunniplaan.meetings_per_course().items():
    print(f"{aine}: {arv}")

print("\nb) Ainete nimekiri õppejõu järgi:")
for oppejoud, ained in tunniplaan.courses_by_teacher().items():
    print(f"{oppejoud}: {', '.join(ained)}")

print("\nc) Kaks lähiajal õpetatavat ainet:")
for aine in tunniplaan.next_two_courses():
    print(aine)

Tunniplaan: 11 õppeainet, 51 sündmust

a) Mitu kohtumist on igas aines:
Programmeerimine: 4
Inseneriinformaatika: 3
Digitaalloogika ja -süsteemid: 6
Rakendusfüüsika: 5
Operatsioonisüsteemide administreerimine: 5
Eesti keel III: 8
Rohetehnoloogiate baaskursus: 6

b) Ainete nimekiri õppejõu järgi:
Maarika Virkunen: Programmeerimine, Rakendusfüüsika
Natalja Ivleva: Programmeerimine
Valeria Juštšenko: Inseneriinformaatika
Žanna Gratšjova: Inseneriinformaatika
Oleg Shvets: Digitaalloogika ja -süsteemid, Operatsioonisüsteemide administreerimine
Monika Jänis: Digitaalloogika ja -süsteemid
Sergei Pavlov: Rakendusfüüsika
Valentina Limonova: Eesti keel III
Dmitri Nikitin: Rohetehnoloogiate baaskursus

c) Kaks lähiajal õpetatavat ainet:
Inseneriinformaatika
Programmeerimine


## 2.8 Kahe lahenduse võrdlus

Allpool võrdlen lühidalt andmestruktuuride lahendust ja OOP lahendust.

| Küsimus | Andmestruktuuride lahendus | OOP lahendus |
|---|---|---|
| Kus hoitakse andmeid? | `dict`, `list`, `tuple` tüüpi andmestruktuurides | Klasside eksemplarides (`Event`, `Course`, `Timetable`) |
| Mis juhtub, kui andmestruktuur muutub? | Tuleb muuta mitut kohta koodis, sest võtmed ja indeksid võivad muutuda | Muudatused saab enamasti koondada klassidesse ja property-meetoditesse |
| Kui lihtne on lisada uut funktsiooni? | Väikese ülesande puhul on lihtne, aga suuremaks minnes muutub loogika hajusaks | Uut funktsiooni on lihtsam lisada klassi meetodina, sest andmed ja käitumine on koos |
| Kumb on loetavam? | Väikese ülesande puhul võib olla lühem | Minu arvates on OOP loetavam, sest klassinimed näitavad kohe, mida andmed tähendavad |
| Millal kasutaksite kumba? | Kasutaksin väikese ja ühekordse ülesande puhul | Kasutaksin OOP-i siis, kui programm kasvab suuremaks ja vajab paremat struktuuri |

In [40]:
@dataclass
class Event:
    """
    summary: str
    dtstart: datetime
    dtend: datetime
    location: str = None # Optional field with default value None
    description: str = None # Optional field with default value None
    """
    summary: str
    dtstart: datetime
    dtend: datetime
    location: str = None
    description: str = None

@dataclass
class Course:
    code: str
    name: str
    teacher: str
    events: list[Event]

@dataclass
class Timetable:
    courses: list[Course]

    @staticmethod
    def from_ical(filename: str):
        with open(filename, 'rb') as f:
            cal = Calendar.from_ical(f.read())
        events = []
        for component in cal.walk():
            if component.name == "VEVENT":
                event = Event(
                    summary=component.get('summary'),
                    dtstart=component.get('dtstart').dt,
                    dtend=component.get('dtend').dt,
                    location=str(component.get('location')) if component.get('location') else None,
                    description=str(component.get('description')) if component.get('description') else None
                )
                events.append(event)
        return events


In [41]:
events = Timetable.from_ical('tunniplaan.ics')
for event in events:
    print(event)

Event(summary=vText(b'RAM0620 - Programmeerimine'), dtstart=datetime.datetime(2026, 2, 7, 8, 45, tzinfo=zoneinfo.ZoneInfo(key='UTC')), dtend=datetime.datetime(2026, 2, 7, 10, 15, tzinfo=zoneinfo.ZoneInfo(key='UTC')), location='VK1-42', description='Õppejõud: Maarika Virkunen')
Event(summary=vText(b'RAM0620 - Programmeerimine'), dtstart=datetime.datetime(2026, 3, 7, 12, 45, tzinfo=zoneinfo.ZoneInfo(key='UTC')), dtend=datetime.datetime(2026, 3, 7, 14, 15, tzinfo=zoneinfo.ZoneInfo(key='UTC')), location='VK1-40', description='Õppejõud: Maarika Virkunen')
Event(summary=vText(b'RAM0620 - Programmeerimine'), dtstart=datetime.datetime(2026, 3, 21, 7, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC')), dtend=datetime.datetime(2026, 3, 21, 10, 15, tzinfo=zoneinfo.ZoneInfo(key='UTC')), location='VK1-28', description='Õppejõud: Natalja Ivleva')
Event(summary=vText(b'RAM0620 - Programmeerimine'), dtstart=datetime.datetime(2026, 4, 4, 10, 0, tzinfo=zoneinfo.ZoneInfo(key='UTC')), dtend=datetime.datetime(2026, 4